# Enum

> Generated from [`08-Enum.py`](08-Enum.py) by `tools/build_notebooks.py` - edit the `.py` (or its `.md` notes), not this notebook. Changes made here are overwritten on the next build.

## Quick notes

### What this covers
The `enum` module — defining a fixed set of named constants, safer and clearer than scattering raw strings or magic numbers through code.

### Key points
- **`class Color(Enum): RED = 1 ...`** — access members as `Color.RED`, with `.name` and `.value`.
- **`auto()`** — auto-assigns values so you don't have to pick numbers yourself.
- **Enum members are singletons** — `current is Status.ACTIVE` works, not just `==`.
- **Iterate** an enum class directly (`for s in Status`) to get all members.
- **Lookup** — `Status(1)` by value, `Status["ACTIVE"]` by name.
- **`IntEnum`** — members also behave like real ints (comparable, addable) — useful when you need enum safety but int compatibility (e.g. priorities).
- **`Flag` / `IntFlag`** — combinable bitwise enums; OR members together (`READ | WRITE`), test with `in`. `IntFlag` also behaves like an int (chmod-style values).
- **Custom methods/properties** — an Enum class can define `@property` and regular methods, using `self.value` for computation.
- **`@unique`** — decorator that forbids two members from sharing the same value (plain `Enum` silently allows duplicates as aliases).
- **`StrEnum`** (3.11+) or the pre-3.11 `class Role(str, Enum)` pattern — members that ARE strings (`isinstance(Role.ADMIN, str)` is `True`, compares equal to raw strings).
- **`_missing_`** — a classmethod hook called when a value doesn't match any member directly; lets you normalize input (e.g. case-insensitive lookup) before falling back to `ValueError`.

### Gotchas (the part worth revising twice)
- Without `@unique`, an `Enum` with two members sharing the same value silently makes the second one an *alias* of the first — no error, easy to miss.
- `IntEnum`/`IntFlag` give up some of Enum's type safety (they behave like plain ints), trading it for numeric compatibility — pick deliberately.
- `Flag` members combine with `|` and check with `in` — don't reach for a list/set of enums when a `Flag` says what you mean more directly.
- `_missing_` returning `None` means "no match" — the normal `ValueError` still gets raised; it's only a hook to normalize *before* failing, not to suppress failure entirely.

### When it matters
Anywhere you'd otherwise use magic strings/numbers for a fixed set of options — statuses, roles, permissions, directions — where typos should be caught early instead of failing silently.

### One-line recap example
```python
from enum import Enum
class Status(Enum):
    ACTIVE = "active"
    CLOSED = "closed"
print(Status.ACTIVE == Status.ACTIVE)   # True, and it's a real singleton, not a raw string
```

In [ ]:
from enum import Enum, IntEnum, auto

## 01. Enum - Introduction

- Enum defines a fixed set of named constants, safer/clearer than raw
  strings or magic numbers scattered through code.

## 02. Defining and Using an Enum

In [ ]:
class Color(Enum):
    RED = 1
    GREEN = 2
    BLUE = 3

print("\n# 02. Defining and Using an Enum")
print(Color.RED)
print(Color.RED.name, "=", Color.RED.value)
print(type(Color.RED))

## 03. Without Enum - the Problem (magic values, no safety)

In [ ]:
def set_status_bad(status):
    if status == "active":  # typo-prone, no autocomplete, no validation
        print("Status set")

print("\n# 03. Without Enum - the Problem")
print("# - Raw strings/ints: typos slip through silently, no fixed set of values.")

## 04. auto() - Auto-Assigned Values

In [ ]:
class Status(Enum):
    PENDING = auto()
    ACTIVE = auto()
    CLOSED = auto()

print("\n# 04. auto()")
for status in Status:
    print(status.name, "=", status.value)

## 05. Comparing Enum Members

In [ ]:
current = Status.ACTIVE
print("current == Status.ACTIVE:", current == Status.ACTIVE)
print("current is Status.ACTIVE:", current is Status.ACTIVE)  # members are singletons

## 06. Iterating Over an Enum

In [ ]:
print([s.name for s in Status])

## 07. Looking Up Members by Value or Name

In [ ]:
print(Status(1))               # by value
print(Status["ACTIVE"])        # by name

## 08. IntEnum - Members That Also Behave Like Ints

In [ ]:
class Priority(IntEnum):
    LOW = 1
    MEDIUM = 2
    HIGH = 3

print("\n# 08. IntEnum")
print(Priority.HIGH > Priority.LOW)   # works like int comparison
print(Priority.HIGH + 1)               # can do int arithmetic too

## 09. Enum in Practice - Replacing Magic Values

In [ ]:
def set_status(status: Status):
    print(f"Status set to {status.name}")

print("\n# 09. Enum in Practice")
set_status(Status.ACTIVE)

## 10. Flag / IntFlag - Combinable Bitwise Enum Members

- Flag members can be OR'd together (|) into a single value, and you can
  test membership with `in`. Each member is a distinct bit.

In [ ]:
from enum import Flag, IntFlag

class Permission(Flag):
    READ = auto()
    WRITE = auto()
    EXECUTE = auto()

print("\n# 10. Flag / IntFlag")
user_perms = Permission.READ | Permission.WRITE
print("combined:", user_perms)
print("has READ:", Permission.READ in user_perms)
print("has EXECUTE:", Permission.EXECUTE in user_perms)

class FilePermission(IntFlag):  # also behaves like an int (e.g. chmod-style)
    READ = 4
    WRITE = 2
    EXECUTE = 1

rwx = FilePermission.READ | FilePermission.WRITE | FilePermission.EXECUTE
print("chmod-style value:", int(rwx))

## 11. Custom Methods and Properties on an Enum

In [ ]:
class Direction(Enum):
    NORTH = (0, 1)
    SOUTH = (0, -1)
    EAST = (1, 0)
    WEST = (-1, 0)

    @property
    def dx(self):
        return self.value[0]

    @property
    def dy(self):
        return self.value[1]

    def opposite(self):
        return {Direction.NORTH: Direction.SOUTH, Direction.SOUTH: Direction.NORTH,
                 Direction.EAST: Direction.WEST, Direction.WEST: Direction.EAST}[self]

print("\n# 11. Custom Methods/Properties on Enum")
print("NORTH delta:", Direction.NORTH.dx, Direction.NORTH.dy)
print("opposite of NORTH:", Direction.NORTH.opposite())

## 12. @unique - Forbid Duplicate Values

In [ ]:
from enum import unique

print("\n# 12. @unique decorator")
try:
    @unique
    class Broken(Enum):
        A = 1
        B = 1  # duplicate value - @unique catches what plain Enum allows as an alias
except ValueError as e:
    print("ValueError:", e)

## 13. StrEnum (3.11+) / Pre-3.11 str+Enum Pattern

In [ ]:
import sys

print("\n# 13. StrEnum / str+Enum Pattern")
if sys.version_info >= (3, 11):
    from enum import StrEnum

    class Role(StrEnum):
        ADMIN = "admin"
        USER = "user"
else:
    class Role(str, Enum):  # pre-3.11 equivalent - members ARE strings
        ADMIN = "admin"
        USER = "user"

print("Role.ADMIN:", Role.ADMIN, "| is a str:", isinstance(Role.ADMIN, str))
print("equals raw string:", Role.ADMIN == "admin")

## 14. _missing_ - Custom Lookup Fallback

- Called when Enum(value) doesn't match any member's value directly -
  lets you normalize input (e.g. case-insensitive lookup) before failing.

In [ ]:
class Size(Enum):
    SMALL = "S"
    MEDIUM = "M"
    LARGE = "L"

    @classmethod
    def _missing_(cls, value):
        if isinstance(value, str):
            for member in cls:
                if member.value.lower() == value.lower():
                    return member
        return None  # returning None -> normal ValueError is raised

print("\n# 14. _missing_ Custom Lookup")
print(Size("s"))          # lowercase, falls back to case-insensitive match
print(Size("m"))          # lowercase, falls back to case-insensitive match
try:
    Size("XL")
except ValueError as e:
    print("ValueError:", e)